# 2026 世界杯数据探索分析\n\n分析比赛数据特征：进球分布、胜平负比例、排名与结果的相关性、预测模型初步评估。

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from scipy import stats

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
plt.style.use('ggplot')
%matplotlib inline

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PROCESSED = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'output'

## 1. 数据加载

In [ ]:
# FIFA 排名
FIFA_RANKINGS = {
    "Argentina": 1, "Spain": 2, "France": 3, "England": 4,
    "Portugal": 5, "Brazil": 6, "Morocco": 7, "Netherlands": 8,
    "Belgium": 9, "Germany": 10, "Croatia": 11, "Colombia": 13,
    "Mexico": 14, "Senegal": 15, "Uruguay": 16, "USA": 17,
    "Japan": 18, "Switzerland": 19, "Iran": 20, "Turkey": 22,
    "Ecuador": 23, "Austria": 24, "South Korea": 25, "Australia": 27,
    "Algeria": 28, "Egypt": 29, "Canada": 30, "Norway": 31,
    "Ivory Coast": 33, "Panama": 34, "Sweden": 38, "Czech Republic": 40,
    "Paraguay": 41, "Scotland": 42, "Tunisia": 45, "DR Congo": 46,
    "Uzbekistan": 50, "Qatar": 56, "Iraq": 57, "Saudi Arabia": 61,
    "Jordan": 63, "Bosnia and Herzegovina": 64, "South Africa": 60,
    "Cape Verde": 67, "Curacao": 82, "Haiti": 83, "New Zealand": 85,
    "Ghana": 73,
}

# 第一轮比赛结果
MATCHES = [
    ("Mexico", "South Africa", 2, 0),
    ("South Korea", "Czech Republic", 2, 1),
    ("Canada", "Bosnia and Herzegovina", 1, 1),
    ("Qatar", "Switzerland", 1, 1),
    ("Brazil", "Morocco", 1, 1),
    ("Haiti", "Scotland", 0, 1),
    ("USA", "Paraguay", 4, 1),
    ("Australia", "Turkey", 2, 0),
    ("Germany", "Curacao", 7, 1),
    ("Ivory Coast", "Ecuador", 1, 0),
    ("Netherlands", "Japan", 2, 2),
    ("Sweden", "Tunisia", 5, 1),
    ("Belgium", "Egypt", 1, 1),
    ("Iran", "New Zealand", 2, 2),
    ("Spain", "Cape Verde", 0, 0),
    ("Saudi Arabia", "Uruguay", 1, 1),
]

df = pd.DataFrame(MATCHES, columns=['home', 'away', 'h_goals', 'a_goals'])
df['total_goals'] = df['h_goals'] + df['a_goals']
df['goal_diff'] = df['h_goals'] - df['a_goals']

def result(row):
    if row['h_goals'] > row['a_goals']: return 'H'
    elif row['h_goals'] < row['a_goals']: return 'A'
    else: return 'D'

df['result'] = df.apply(result, axis=1)
df['home_rank'] = df['home'].map(FIFA_RANKINGS)
df['away_rank'] = df['away'].map(FIFA_RANKINGS)
df['rank_diff'] = df['home_rank'] - df['away_rank']

print(f"总场次: {len(df)}")
print(f"场均进球: {df['total_goals'].mean():.2f}")
df.head(10)

## 2. 赛果分布

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 胜平负
counts = df['result'].value_counts()
labels = {'H': '主胜', 'D': '平局', 'A': '客胜'}
ax = axes[0]
bars = ax.bar([labels.get(k, k) for k in counts.index], counts.values,
              color=['#2ecc71', '#95a5a6', '#e74c3c'])
ax.set_title('胜平负分布')
ax.set_ylabel('场次')
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{v}场 ({v/16*100:.0f}%)', ha='center', fontsize=11)

# 总进球分布
ax = axes[1]
ax.hist(df['total_goals'], bins=range(0, 10), edgecolor='white', color='#3498db', alpha=0.8)
ax.set_title('总进球数分布')
ax.set_xlabel('进球数')
ax.set_ylabel('场次')
ax.set_xticks(range(0, 9))

# 分差分布
ax = axes[2]
ax.hist(df['goal_diff'], bins=range(-5, 7), edgecolor='white', color='#9b59b6', alpha=0.8)
ax.set_title('净胜球分布')
ax.set_xlabel('主队净胜球')
ax.set_ylabel('场次')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'match_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 排名与结果的关系

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 排名差 vs 净胜球
ax = axes[0]
colors = ['#2ecc71' if d > 0 else '#e74c3c' if d < 0 else '#95a5a6' for d in df['goal_diff']]
ax.scatter(df['rank_diff'], df['goal_diff'], c=colors, s=100, edgecolors='black', linewidth=0.5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('排名差 (主队排名 - 客队排名)')
ax.set_ylabel('净胜球')
ax.set_title('FIFA排名差 vs 比赛净胜球')

# 标注解读
ax.text(0.98, 0.02, '右下 = 强主队大胜\n左上 = 强客队大胜',
        transform=ax.transAxes, fontsize=9, ha='right', va='bottom',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 排名差 vs 结果
ax = axes[1]
result_order = ['A', 'D', 'H']
colors_map = {'H': '#2ecc71', 'D': '#95a5a6', 'A': '#e74c3c'}
for res in result_order:
    subset = df[df['result'] == res]
    ax.scatter([res] * len(subset), subset['rank_diff'],
              color=colors_map[res], s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('排名差')
ax.set_title('赛果 vs 排名差')
ax.set_xticklabels(['客胜', '平局', '主胜'])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ranking_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# 统计
rank_favored_correct = 0
for _, row in df.iterrows():
    favored_home = row['rank_diff'] < -10  # 主队排名明显更高
    favored_away = row['rank_diff'] > 10   # 客队排名明显更高
    if favored_home and row['result'] == 'H':
        rank_favored_correct += 1
    elif favored_away and row['result'] == 'A':
        rank_favored_correct += 1

print(f"排名明显优势方获胜: {rank_favored_correct}/{len(df)} = {rank_favored_correct/len(df)*100:.1f}%")

## 4. 进球泊松分布检验

In [ ]:
from scipy.stats import poisson

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 主队进球 vs 泊松拟合
all_home_goals = df['h_goals'].values
mean_home = all_home_goals.mean()

ax = axes[0]
max_g = 8
observed = np.bincount(all_home_goals, minlength=max_g+1)[:max_g+1]
expected = np.array([poisson.pmf(k, mean_home) * len(df) for k in range(max_g+1)])
x = np.arange(max_g+1)
width = 0.35
ax.bar(x - width/2, observed, width, label='实际', color='#3498db', alpha=0.8)
ax.bar(x + width/2, expected, width, label=f'泊松(λ={mean_home:.1f})', color='#e74c3c', alpha=0.6)
ax.set_title('主队进球分布 vs 泊松拟合')
ax.set_xlabel('进球数')
ax.legend()

# 客队进球 vs 泊松拟合
all_away_goals = df['a_goals'].values
mean_away = all_away_goals.mean()

ax = axes[1]
observed_a = np.bincount(all_away_goals, minlength=max_g+1)[:max_g+1]
expected_a = np.array([poisson.pmf(k, mean_away) * len(df) for k in range(max_g+1)])
ax.bar(x - width/2, observed_a, width, label='实际', color='#9b59b6', alpha=0.8)
ax.bar(x + width/2, expected_a, width, label=f'泊松(λ={mean_away:.1f})', color='#e74c3c', alpha=0.6)
ax.set_title('客队进球分布 vs 泊松拟合')
ax.set_xlabel('进球数')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'poisson_fit.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"主队场均进球: {mean_home:.2f}, 泊松检验 p-value: 待更大样本")
print(f"客队场均进球: {mean_away:.2f}")

## 5. 预测模型对比（基于回测结果）

In [ ]:
# 读取回测结果
backtest_path = OUTPUT_DIR / 'backtest_summary.csv'
if backtest_path.exists():
    summary = pd.read_csv(backtest_path, encoding='utf-8-sig')
else:
    # 用回测脚本实时数据
    summary = pd.DataFrame([
        {"method": "随机均匀(33%)", "win_accuracy": 33.0, "std": 11.5},
        {"method": "随机加权(45/25/30)", "win_accuracy": 33.0, "std": 12.1},
        {"method": "FIFA排名启发式", "win_accuracy": 43.8, "std": None},
        {"method": "泊松模型", "win_accuracy": 50.0, "std": None},
        {"method": "ELO概率模型", "win_accuracy": 39.3, "std": 10.0},
        {"method": "易经随机卦", "win_accuracy": 32.7, "std": 12.8},
        {"method": "八卦随机", "win_accuracy": 30.4, "std": 11.4},
        {"method": "博彩盘口", "win_accuracy": 37.5, "std": None},
        {"method": "赔率校准泊松", "win_accuracy": 31.2, "std": None},
    ])

summary = summary.sort_values('win_accuracy', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))

colors = []
for m in summary['method']:
    if '随机' in m or '易经' in m or '八卦' in m:
        colors.append('#bdc3c7')  # 随机基线 - 灰色
    elif '赔率' in m or '盘口' in m:
        colors.append('#f39c12')  # 市场数据 - 橙色
    else:
        colors.append('#2ecc71')  # 模型方法 - 绿色

bars = ax.barh(summary['method'], summary['win_accuracy'], color=colors, edgecolor='white')

# 标注
for bar, v, std_v in zip(bars, summary['win_accuracy'], summary['std']):
    label = f'{v:.1f}%'
    if std_v and std_v > 0:
        label += f' ±{std_v:.1f}%'
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=10)

ax.axvline(x=33.3, color='gray', linestyle='--', alpha=0.5, label='随机基线 (33.3%)')
ax.set_xlabel('胜平负准确率 (%)')
ax.set_title('预测方法对比 — 2026世界杯第一轮16场回测')
ax.set_xlim(0, 70)
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 赔率市场效率分析

In [ ]:
# 博彩赔率隐含概率 vs 实际结果
FALLBACK_ODDS = [
    ("Mexico", "South Africa", 1.42, 4.40, 7.27, 2, 0),
    ("South Korea", "Czech Republic", 2.57, 2.95, 2.84, 2, 1),
    ("Canada", "Bosnia and Herzegovina", 1.76, 3.51, 4.62, 1, 1),
    ("Qatar", "Switzerland", 13.87, 6.22, 1.26, 1, 1),
    ("Brazil", "Morocco", 1.62, 3.77, 5.42, 1, 1),
    ("Haiti", "Scotland", 6.82, 4.60, 1.44, 0, 1),
    ("USA", "Paraguay", 1.93, 3.43, 3.82, 4, 1),
    ("Australia", "Turkey", 4.68, 3.68, 1.66, 2, 0),
    ("Germany", "Curacao", 1.03, 16.50, 42.00, 7, 1),
    ("Ivory Coast", "Ecuador", 3.53, 2.77, 2.36, 1, 0),
    ("Netherlands", "Japan", 1.91, 3.50, 3.78, 2, 2),
    ("Sweden", "Tunisia", 1.86, 3.30, 4.25, 5, 1),
    ("Belgium", "Egypt", 1.58, 3.98, 5.37, 1, 1),
    ("Iran", "New Zealand", 1.86, 3.57, 4.39, 2, 2),
    ("Spain", "Cape Verde", 1.09, 9.85, 29.50, 0, 0),
    ("Saudi Arabia", "Uruguay", 7.07, 4.35, 1.39, 1, 1),
]

rows = []
for h, a, oh, od, oa, hs, as_ in FALLBACK_ODDS:
    imp_h = 1/oh
    imp_d = 1/od
    imp_a = 1/oa
    overround = imp_h + imp_d + imp_a
    rows.append({
        'match': f'{h} vs {a}',
        'prob_h': imp_h/overround*100,
        'prob_d': imp_d/overround*100,
        'prob_a': imp_a/overround*100,
        'predicted': 'H' if max(imp_h, imp_d, imp_a) == imp_h else ('D' if max(imp_h, imp_d, imp_a) == imp_d else 'A'),
        'actual': 'H' if hs > as_ else ('D' if hs == as_ else 'A'),
        'correct': ('H' if hs > as_ else ('D' if hs == as_ else 'A')) == ('H' if max(imp_h, imp_d, imp_a) == imp_h else ('D' if max(imp_h, imp_d, imp_a) == imp_d else 'A')),
        'overround': (overround - 1) * 100
    })

odds_df = pd.DataFrame(rows)
print(f"博彩盘口准确率: {odds_df['correct'].sum()}/{len(odds_df)} = {odds_df['correct'].mean()*100:.1f}%")
print(f"庄家平均抽水: {odds_df['overround'].mean():.1f}%")
print()
print("赔率错误分类的场次:")
odds_df[~odds_df['correct']][['match', 'predicted', 'actual', 'prob_h', 'prob_d', 'prob_a']]

## 7. 结论与下一步

1. **泊松模型 50% 准确率**在 16 场小样本上表现最好，但需要更多比赛验证
2. **赔率校准反而降低准确率**——盘口隐含概率与泊松结合的偏差可能来自赔率本身的信息噪声
3. **博彩盘口 37.5%** 远低于理论值，16 场期间冷门频出（世界杯小组赛首轮波动大）
4. **玄学基线（易经/八卦）在 30-33%**，刚好等于随机，符合预期

下一步方向：
- 接入更多轮次数据扩大样本
- 引入球队进攻/防守强度参数（Dixon-Coles 模型）
- 用历史世界杯数据训练泊松回归系数